# Making It Production-Ready

In the previous notebook we built a 3-agent DevOps pipeline. It works — but it has problems you'd hit immediately in production. This notebook fixes them, one at a time.

| Section | The Problem | The Fix | CrewAI API |
|---------|------------|---------|------------|
| 1 | Agent output is raw text — parsing it is fragile | **Structured Output** | `output_pydantic=Model` |
| 2 | Bad or vague output passes through unchecked | **Code Guardrail** | `guardrail=validate_fn` |
| 3 | Writing validation functions for everything is tedious | **No-Code Guardrail** | `guardrail="plain English"` |

In [ ]:
import os
from typing import Any, Tuple

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai.tasks.task_output import TaskOutput
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

llm = LLM(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

In [ ]:
LOG_INPUT = """
[2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment
[2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state
[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start
[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff
[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
[2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints
[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated
"""

log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    verbose=True,
)

---

## 1. Structured Output

### The Problem: Raw Text Is Fragile

In Notebook 1, our agent returns a wall of markdown text. Let's run it and try to extract specific fields.

In [ ]:
v1_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
)

v1_crew = Crew(
    agents=[log_analyzer],
    tasks=[v1_task],
    process=Process.sequential,
    verbose=False,
)

v1_result = v1_crew.kickoff(inputs={"log_data": LOG_INPUT})

In [ ]:
print(v1_result.raw)

In [ ]:
# Want the root cause? Parse the string:
lines = v1_result.raw.split("\n")
for line in lines:
    if "root cause" in line.lower():
        print(f"Found: {line}")
        break

# Want the number of errors? Count manually in free-form text?
# Want to pass typed data to the next agent? Impossible.
# What if the agent formats it differently next run? Everything breaks.

### The Fix: `output_pydantic`

Define a Pydantic model describing the shape of the output you want. Add `output_pydantic=Model` to the task. CrewAI forces the agent to return data matching that schema.

In [ ]:
class LogAnalysisReport(BaseModel):
    primary_issue: str = Field(description="One-line description of the main issue")
    root_cause: str = Field(description="Root cause analysis based on log evidence")
    errors: list[str] = Field(description="All errors found in the log")
    affected_components: list[str] = Field(description="System components affected")
    timeline: list[str] = Field(description="Sequence of events leading to failure")

In [ ]:
structured_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    agent=log_analyzer,
)

structured_crew = Crew(
    agents=[log_analyzer],
    tasks=[structured_task],
    process=Process.sequential,
    verbose=False,
)

structured_result = structured_crew.kickoff(inputs={"log_data": LOG_INPUT})

In [ ]:
report = structured_result.pydantic

print(f"Primary issue: {report.primary_issue}")

In [ ]:
print(f"Root cause: {report.root_cause}")

In [ ]:
print(f"\nErrors found: {len(report.errors)}")
for error in report.errors:
    print(f"  - {error}")

In [ ]:
print(f"\nAffected components: {report.affected_components}")

In [ ]:
print(f"\nTimeline:")
for event in report.timeline:
    print(f"  - {event}")

In [ ]:
print(f"\nFull JSON:\n{report.model_dump_json(indent=2)}")

Same agent. Same log. One parameter changed everything — `output_pydantic=LogAnalysisReport`. Now every field is typed, accessible, and guaranteed to be there.

---

## 2. Code Guardrail

### The Problem: Bad Output Passes Through

Structured output guarantees the *shape*, but not the *quality*. What if the agent returns a report with zero errors identified? With no guardrail, that bad output flows straight to the next agent unchecked.

Here's a log where everything is labeled INFO — no explicit ERROR lines. The agent sees "no errors" and returns an empty `errors` list. The shape is valid, but the content is useless.

In [ ]:
TRICKY_LOG_INPUT = """
[2024-01-15 09:01:22.100] INFO: Cron job scheduled-cleanup started
[2024-01-15 09:01:23.200] INFO: Connected to database cluster (primary)
[2024-01-15 09:01:24.300] INFO: Processing batch 1 of 1
[2024-01-15 09:01:25.400] INFO: 0 records processed
[2024-01-15 09:01:26.500] INFO: Disk usage at 94%
[2024-01-15 09:01:27.600] INFO: Cron job scheduled-cleanup completed successfully
"""

unguarded_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    agent=log_analyzer,
)

unguarded_crew = Crew(
    agents=[log_analyzer],
    tasks=[unguarded_task],
    process=Process.sequential,
    verbose=False,
)

unguarded_result = unguarded_crew.kickoff(inputs={"log_data": TRICKY_LOG_INPUT})

In [ ]:
report = unguarded_result.pydantic
print(f"Errors found: {len(report.errors)}")
print(f"Errors: {report.errors}")
print(f"Root cause: {report.root_cause}")

passed = len(report.errors) > 0
print(f"\nWould pass guardrail? {passed}")
if not passed:
    print("The agent saw all-INFO logs and returned zero errors.")
    print("But '0 records processed' and '94% disk' ARE problems.")
    print("Without a guardrail, this empty report goes straight to the next agent.")

### The Fix: Add a Guardrail

A guardrail is a function that validates the output *before* it's accepted. It returns:
- `(True, data)` — output is good, pass it through
- `(False, "reason")` — output is rejected, agent retries automatically

Now the same log, same agent — but with the guardrail. When the agent returns zero errors, the guardrail rejects it and the agent retries, digging deeper into the INFO messages to find the real issues. Run with `verbose=True` to see the retry in action.

In [ ]:
def validate_log_analysis(result: TaskOutput) -> Tuple[bool, Any]:
    report = result.pydantic
    if not report or not report.errors:
        return (False, "Must identify at least one error")
    return (True, report)

In [ ]:
guarded_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    guardrail=validate_log_analysis,
    agent=log_analyzer,
)

guarded_crew = Crew(
    agents=[log_analyzer],
    tasks=[guarded_task],
    process=Process.sequential,
    verbose=True,
)

guarded_result = guarded_crew.kickoff(inputs={"log_data": TRICKY_LOG_INPUT})

In [ ]:
report = guarded_result.pydantic
print(f"Errors found: {len(report.errors)}")
for error in report.errors:
    print(f"  - {error}")
print(f"\nRoot cause: {report.root_cause}")

---

## 3. No-Code Guardrail

Writing validation functions for everything is tedious. For simpler checks, you can pass a **plain English string** as the guardrail. CrewAI uses an LLM to evaluate whether the output meets your criteria.

In [ ]:
solution_specialist = Agent(
    role="DevOps Solution Specialist",
    goal="Provide clear, actionable solutions with step-by-step instructions",
    llm=llm,
    backstory="""You are a DevOps solutions architect who specializes in creating 
    reliable, step-by-step remediation plans for infrastructure issues.""",
    verbose=True,
)

solution_task = Task(
    description="Provide a solution for the following issue: {issue}",
    expected_output="A remediation plan with specific commands",
    guardrail="The solution must include at least 3 specific, copy-pasteable shell commands. "
    "Reject if it only contains general advice without concrete commands.",
    agent=solution_specialist,
)

solution_crew = Crew(
    agents=[solution_specialist],
    tasks=[solution_task],
    process=Process.sequential,
    verbose=True,
)

solution_result = solution_crew.kickoff(
    inputs={"issue": "Kubernetes pods failing with ImagePullBackOff due to missing registry credentials"}
)
print(f"\nFinal solution:\n{solution_result.raw}")

---

## 4. The Full Pipeline

Everything from this notebook in one cell. Three agents, three tasks, every production feature working together:

- **Structured output** on the analysis task — typed `LogAnalysisReport` instead of raw text.
- **Code guardrail** on the analysis task — retries if no errors are identified.
- **Context passing** — each downstream task receives output from the tasks before it.
- **No-code guardrail** on the solution task — plain English check for concrete commands.
- **`output_file`** — all three task results saved to disk.

In [ ]:
os.makedirs("task_outputs", exist_ok=True)

log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    verbose=True,
)

issue_investigator = Agent(
    role="DevOps Issue Investigator",
    goal="Investigate identified issues by searching documentation, forums, and known solutions",
    llm=llm,
    backstory="""You are a DevOps troubleshooting specialist who excels at quickly 
    finding solutions to technical problems. You know how to identify reliable sources 
    and gather comprehensive information about error patterns and their solutions.""",
    verbose=True,
)

solution_specialist = Agent(
    role="DevOps Solution Specialist",
    goal="Provide clear, actionable solutions with step-by-step instructions",
    llm=llm,
    backstory="""You are a DevOps solutions architect who specializes in creating 
    reliable, step-by-step remediation plans for infrastructure issues.""",
    verbose=True,
)

def validate_log_analysis(result: TaskOutput) -> Tuple[bool, Any]:
    report = result.pydantic
    if not report or not report.errors:
        return (False, "Must identify at least one error")
    return (True, report)

analyze_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    guardrail=validate_log_analysis,
    agent=log_analyzer,
    output_file="task_outputs/log_analysis.json",
)

investigate_task = Task(
    description="""Based on the log analysis findings, investigate the identified issue.
    
    Your investigation should:
    1. Identify common causes and scenarios for this type of issue
    2. Find known solutions and best practices
    3. Gather information about proven fixes and workarounds""",
    expected_output="""A comprehensive investigation report including:
    - Common causes ranked by likelihood
    - Known solutions and best practices
    - Recommended fixes and workarounds""",
    agent=issue_investigator,
    context=[analyze_task],
    output_file="task_outputs/investigation_report.md",
)

solution_task = Task(
    description="""Based on the log analysis and investigation findings, provide a complete solution.
    
    Your solution should:
    1. Create a step-by-step remediation plan with specific commands
    2. Provide verification steps to confirm the fix
    3. Suggest monitoring and prevention measures""",
    expected_output="A detailed remediation plan with step-by-step commands",
    guardrail="The solution must include at least 3 specific, copy-pasteable shell commands. "
    "Reject if it only contains general advice without concrete commands.",
    agent=solution_specialist,
    context=[analyze_task, investigate_task],
    output_file="task_outputs/solution_plan.md",
)

pipeline_crew = Crew(
    agents=[log_analyzer, issue_investigator, solution_specialist],
    tasks=[analyze_task, investigate_task, solution_task],
    process=Process.sequential,
    verbose=True,
)

result = pipeline_crew.kickoff(inputs={"log_data": LOG_INPUT})

In [ ]:
report = analyze_task.output.pydantic
print("STRUCTURED ANALYSIS (from guardrailed task):")
print(f"  Primary issue: {report.primary_issue}")
print(f"  Root cause: {report.root_cause}")
print(f"  Errors: {report.errors}")
print(f"  Affected components: {report.affected_components}")
print(f"  Timeline: {report.timeline}")

print(f"\nSOLUTION (from no-code guardrailed task):")
print(result.raw)

---

## Recap

| Feature | Without (Notebook 1) | With (This Notebook) | One-Line Change |
|---------|---------------------|---------------------|----------------|
| **Structured Output** | Raw markdown text, fragile parsing | Typed Python objects, guaranteed schema | `output_pydantic=Model` |
| **Code Guardrail** | Bad output passes through silently | Agent retries until output is valid | `guardrail=validate_fn` |
| **No-Code Guardrail** | Write validation code for every check | Describe the check in plain English | `guardrail="must have 3 commands"` |
| **Full Pipeline** | Features used in isolation | All features combined in one crew | Section 4 |